# 4.5 — Pre-scale spectra (arms / same-time groups)

Align flux across spectrograph arms (e.g. XSHooter UVB/VIS/NIR) and same-time epochs **before** mangling (NB5).

- Default **`scale_only`**: separate files, common flux level.
- Optional **`merge_join`**: one merged spectrum per group (set in JSON or `pipeline_config`).

Diagnostics are saved under `Outputs/<SN>/spec_scale_diagnostics/` (open `index.html` after the run).

In [1]:
import os
import sys

COCO_PATH = "/Users/ravkaur/Desktop/research/kilonova-SED/PyCoCo_templates/"
OUTPUT_DIR = os.path.join(COCO_PATH, "Outputs/")

sys.path.insert(0, os.path.join(COCO_PATH, "Codes"))
import pipeline_config as pconf
from spectra_pre_scale import (
    run_prescale_pipeline,
    suggest_scale_groups,
    load_spec_list,
    write_scale_groups_template,
)

snname = pconf.SNNAME_DEFAULT  # e.g. "AT2017gfo"

# Override global default: scale_only | merge_join
SPEC_SCALE_OUTPUT_MODE = pconf.SPEC_SCALE_OUTPUT_MODE
SAME_TIME_MINUTES = pconf.SPEC_SCALE_SAME_TIME_MINUTES
WRITE_DIAGNOSTICS = pconf.SPEC_SCALE_SAVE_DIAGNOSTICS

### Optional — edit groups JSON

On first run, a template is written to `Outputs/<SN>/<SN>_spec_scale_groups.json` from MJD clustering.
Edit **`members`**, **`merge_order`** (`uvb`, `vis`, `nir`), and per-group **`output_mode`** before re-running the scaling cell.

In [2]:
list_path = pconf.smoothed_spec_list_path(COCO_PATH, snname)
entries = load_spec_list(list_path)
groups_path = pconf.spec_scale_groups_json_path(OUTPUT_DIR, snname)

if not os.path.isfile(groups_path):
    suggested = suggest_scale_groups(entries, same_time_minutes=SAME_TIME_MINUTES)
    write_scale_groups_template(groups_path, suggested, default_mode=SPEC_SCALE_OUTPUT_MODE)
    print("Wrote template:", groups_path)
    print("Groups:", len(suggested))
else:
    print("Using existing groups file:", groups_path)

Using existing groups file: /Users/ravkaur/Desktop/research/kilonova-SED/PyCoCo_templates/Outputs/AT2017gfo/AT2017gfo_spec_scale_groups.json


In [ ]:
diag_dir = pconf.spec_scale_diagnostics_dir(OUTPUT_DIR, snname) if WRITE_DIAGNOSTICS else None

report = run_prescale_pipeline(
    snname=snname,
    coco_path=COCO_PATH,
    output_dir=OUTPUT_DIR,
    groups_json=groups_path,
    default_output_mode=SPEC_SCALE_OUTPUT_MODE,
    same_time_minutes=SAME_TIME_MINUTES,
    wl_tol_a=pconf.SPEC_SCALE_OVERLAP_WL_TOL_A,
    gap_log10=pconf.SPEC_SCALE_GAP_LOG10,
    merge_gap_policy=pconf.SPEC_SCALE_MERGE_GAP_POLICY,
    write_diagnostics=WRITE_DIAGNOSTICS,
    diagnostics_dir=diag_dir,
)

print("Prescaled list:", pconf.prescaled_spec_list_path(COCO_PATH, snname))
print("Report:", pconf.spec_scale_report_json_path(OUTPUT_DIR, snname))
print("Groups processed:", len(report.groups))
print("Ungrouped copied:", len(report.ungrouped))
if diag_dir:
    print("Diagnostics index:", os.path.join(diag_dir, "index.html"))